# Flowline2D Model - Basic Usage

This notebook demonstrates basic usage of the flowline2d glacier model using synthetic data. We'll show how to:

1. Set up model configuration
2. Create synthetic glacier geometry
3. Configure climate forcing
4. Run the model
5. Visualize results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add the src directory to the path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from flowline.flowline2d import (
    flowline2d, 
    FlowlineConfig, 
    FlowlineGeometry, 
    TemperaturePrecipitationForcing,
    DirectMassBalanceForcing
)

# Set up plotting
plt.style.use('default')
%matplotlib inline

## 1. Create Synthetic Glacier Geometry

First, we'll create a synthetic glacier valley with realistic bed topography and width profile.

In [ ]:
# Define glacier geometry
x_max = 15000  # Maximum distance (m)
x_gr = np.linspace(0, x_max, 100)  # Distance along flowline

# Create a realistic bed elevation profile (higher at head, lower at terminus)
zb_gr = 3000 - 1500 * (x_gr / x_max)**0.7  # Bed elevation (m)

# Create a width profile (narrower at head, wider at terminus)
w_geom = 200 + 800 * (x_gr / x_max)**0.5  # Width (m)

# Create initial ice thickness profile
x_init = x_gr.copy()
h_init = np.maximum(0, 200 * np.exp(-2 * x_gr / x_max) * (1 - x_gr / x_max))  # Initial thickness (m)

# Visualize the geometry
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8))

ax1.plot(x_gr/1000, zb_gr, 'k-', linewidth=2, label='Bed elevation')
ax1.fill_between(x_gr/1000, zb_gr, zb_gr + h_init, alpha=0.6, color='lightblue', label='Initial ice')
ax1.set_ylabel('Elevation (m)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title('Glacier Profile')

ax2.plot(x_gr/1000, w_geom, 'b-', linewidth=2)
ax2.set_ylabel('Width (m)')
ax2.grid(True, alpha=0.3)
ax2.set_title('Glacier Width')

ax3.plot(x_gr/1000, h_init, 'r-', linewidth=2)
ax3.set_ylabel('Ice thickness (m)')
ax3.set_xlabel('Distance (km)')
ax3.grid(True, alpha=0.3)
ax3.set_title('Initial Ice Thickness')

plt.tight_layout()
plt.show()

print(f"Glacier length: {x_max/1000:.1f} km")
print(f"Elevation range: {zb_gr.min():.0f} - {zb_gr.max():.0f} m")
print(f"Width range: {w_geom.min():.0f} - {w_geom.max():.0f} m")
print(f"Max initial thickness: {h_init.max():.0f} m")

## 2. Example 1: Temperature-Precipitation Forcing

Let's run the model with temperature and precipitation forcing, including some climate variability.

In [ ]:
# Set up model configuration
config = FlowlineConfig(
    delx=100,           # Grid spacing (m)
    delt=0.01,          # Time step (years)
    ts=0,               # Start time (years)
    tf=500,             # End time (years)
    deltout=1,          # Output frequency (years)
    min_thick=1,        # Minimum thickness at terminus (m)
)

# Create geometry object
geometry = FlowlineGeometry(
    x_gr=x_gr,
    zb_gr=zb_gr, 
    w_geom=w_geom,
    x_init=x_init,
    h_init=h_init
)

# Create climate forcing with some variability
nyears = int(config.tf - config.ts)
np.random.seed(42)  # For reproducible results

# Base climate
T0 = 5.0    # Base temperature at sea level (°C)
P0 = 2.0    # Base precipitation (m/yr)

# Add some climate variability
T_anomaly = np.random.normal(0, 0.5, nyears)  # Temperature anomalies
P_anomaly = np.random.normal(0, 0.2, nyears)  # Precipitation anomalies

# Add a warming trend in the last 100 years
warming_trend = np.zeros(nyears)
warming_trend[-100:] = np.linspace(0, 2, 100)  # 2°C warming over 100 years

forcing = TemperaturePrecipitationForcing(
    T0=T0,
    P0=P0,
    sigT=1.0,           # Temperature sensitivity
    sigP=1.0,           # Precipitation sensitivity
    T=T_anomaly,        # Temperature anomalies
    P=P_anomaly,        # Precipitation anomalies
    temp=warming_trend, # Temperature trend
    mu=0.65,            # Melt factor (m/yr/°C)
    gamma=6.5e-3,       # Lapse rate (°C/m)
    ts=config.ts,
    tf=config.tf
)

print("Model configuration:")
print(f"  Grid spacing: {config.delx} m")
print(f"  Time step: {config.delt} years")
print(f"  Simulation period: {config.ts} - {config.tf} years")
print(f"  Base temperature: {T0}°C")
print(f"  Base precipitation: {P0} m/yr")

In [ ]:
# Initialize and run the model
print("Running model with temperature-precipitation forcing...")
model1 = flowline2d(config=config, geometry=geometry, forcing=forcing)
result1 = model1.run()

if result1.no_error:
    print("✓ Model run completed successfully!")
    print(f"  Final glacier length: {result1.edge[-1]/1000:.2f} km")
    print(f"  Final glacier area: {result1.area[-1]/1e6:.2f} km²")
    print(f"  Final max thickness: {result1.h[-1,:].max():.1f} m")
else:
    print("✗ Model run failed!")

In [ ]:
# Plot the results using the built-in plotting method
fig, ax = result1.plot(smooth=10)
fig.suptitle('Temperature-Precipitation Forcing Results', fontsize=14, y=0.98)
plt.show()

## 3. Example 2: Direct Mass Balance Forcing

Now let's run the model with direct mass balance forcing to show the difference.

In [ ]:
# Create a simple mass balance profile (linear with elevation)
# Higher elevations have positive mass balance, lower elevations negative
ela_elevation = 2200  # Equilibrium line altitude (m)
mass_balance_gradient = 0.005  # Mass balance gradient (m/yr per m elevation)

# Calculate base mass balance as function of bed elevation
b0 = mass_balance_gradient * (zb_gr - ela_elevation)

# Add some mass balance variability over time
np.random.seed(123)
b_anomaly = np.random.normal(0, 0.3, nyears)  # Mass balance anomalies

# Add a negative trend (more negative mass balance over time)
b_trend = np.zeros(nyears)
b_trend[-100:] = np.linspace(0, -0.5, 100)  # -0.5 m/yr trend over 100 years

forcing2 = DirectMassBalanceForcing(
    b0=b0,              # Base mass balance profile
    bp=b_anomaly,       # Mass balance perturbations
    bal=b_trend,        # Mass balance trend
    sigb=1.0,           # Mass balance sensitivity
    ts=config.ts,
    tf=config.tf
)

# Visualize the mass balance profile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(b0, zb_gr, 'g-', linewidth=2)
ax1.axvline(0, color='k', linestyle='--', alpha=0.5)
ax1.axhline(ela_elevation, color='r', linestyle='--', alpha=0.5, label=f'ELA = {ela_elevation} m')
ax1.set_xlabel('Mass balance (m/yr)')
ax1.set_ylabel('Elevation (m)')
ax1.set_title('Base Mass Balance Profile')
ax1.grid(True, alpha=0.3)
ax1.legend()

years = np.arange(nyears)
ax2.plot(years, b_anomaly, 'b-', alpha=0.7, linewidth=1, label='Anomalies')
ax2.plot(years, b_trend, 'r-', linewidth=2, label='Trend')
ax2.plot(years, b_anomaly + b_trend, 'k-', linewidth=2, label='Total')
ax2.set_xlabel('Year')
ax2.set_ylabel('Mass balance anomaly (m/yr)')
ax2.set_title('Mass Balance Forcing Time Series')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"ELA elevation: {ela_elevation} m")
print(f"Mass balance gradient: {mass_balance_gradient*1000:.1f} mm/yr per m elevation")
print(f"Mass balance range: {b0.min():.2f} to {b0.max():.2f} m/yr")

In [ ]:
# Run the model with direct mass balance forcing
print("Running model with direct mass balance forcing...")
model2 = flowline2d(config=config, geometry=geometry, forcing=forcing2)
result2 = model2.run()

if result2.no_error:
    print("✓ Model run completed successfully!")
    print(f"  Final glacier length: {result2.edge[-1]/1000:.2f} km")
    print(f"  Final glacier area: {result2.area[-1]/1e6:.2f} km²")
    print(f"  Final max thickness: {result2.h[-1,:].max():.1f} m")
else:
    print("✗ Model run failed!")

In [ ]:
# Plot the results
fig, ax = result2.plot(smooth=10)
fig.suptitle('Direct Mass Balance Forcing Results', fontsize=14, y=0.98)
plt.show()

## 4. Compare Results

Let's compare the two different forcing methods side by side.

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Glacier length comparison
axes[0,0].plot(result1.t, result1.edge/1000, 'b-', linewidth=2, label='T-P forcing')
axes[0,0].plot(result2.t, result2.edge/1000, 'r-', linewidth=2, label='Direct MB forcing')
axes[0,0].set_ylabel('Length (km)')
axes[0,0].set_title('Glacier Length Evolution')
axes[0,0].grid(True, alpha=0.3)
axes[0,0].legend()

# Glacier area comparison
axes[0,1].plot(result1.t, result1.area/1e6, 'b-', linewidth=2, label='T-P forcing')
axes[0,1].plot(result2.t, result2.area/1e6, 'r-', linewidth=2, label='Direct MB forcing')
axes[0,1].set_ylabel('Area (km²)')
axes[0,1].set_title('Glacier Area Evolution')
axes[0,1].grid(True, alpha=0.3)
axes[0,1].legend()

# Mass balance comparison
axes[0,2].plot(result1.t, result1.gwb/result1.area, 'b-', linewidth=2, label='T-P forcing')
axes[0,2].plot(result2.t, result2.gwb/result2.area, 'r-', linewidth=2, label='Direct MB forcing')
axes[0,2].set_ylabel('Specific MB (m/yr)')
axes[0,2].set_title('Specific Mass Balance')
axes[0,2].grid(True, alpha=0.3)
axes[0,2].legend()

# Final glacier profiles
pad = 20
edge1 = int(result1.edge_idx[-1]) + pad
edge2 = int(result2.edge_idx[-1]) + pad
max_edge = max(edge1, edge2)

x_plot = result1.x[:max_edge]
zb_plot = result1.zb[:max_edge]
h1_plot = np.zeros(max_edge)
h2_plot = np.zeros(max_edge)
h1_plot[:len(result1.h[-1,:])] = result1.h[-1,:]
h2_plot[:len(result2.h[-1,:])] = result2.h[-1,:]

axes[1,0].plot(x_plot/1000, zb_plot, 'k-', linewidth=2, label='Bed')
axes[1,0].fill_between(x_plot/1000, zb_plot, zb_plot + h1_plot[:max_edge], 
                      alpha=0.6, color='blue', label='T-P forcing')
axes[1,0].fill_between(x_plot/1000, zb_plot, zb_plot + h2_plot[:max_edge], 
                      alpha=0.4, color='red', label='Direct MB forcing')
axes[1,0].set_xlabel('Distance (km)')
axes[1,0].set_ylabel('Elevation (m)')
axes[1,0].set_title('Final Glacier Profiles')
axes[1,0].grid(True, alpha=0.3)
axes[1,0].legend()

# Max thickness evolution
axes[1,1].plot(result1.t, result1.h.max(axis=1), 'b-', linewidth=2, label='T-P forcing')
axes[1,1].plot(result2.t, result2.h.max(axis=1), 'r-', linewidth=2, label='Direct MB forcing')
axes[1,1].set_xlabel('Time (years)')
axes[1,1].set_ylabel('Max thickness (m)')
axes[1,1].set_title('Maximum Ice Thickness')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].legend()

# ELA evolution
axes[1,2].plot(result1.t, result1.ela, 'b-', linewidth=2, label='T-P forcing')
axes[1,2].plot(result2.t, result2.ela, 'r-', linewidth=2, label='Direct MB forcing')
axes[1,2].set_xlabel('Time (years)')
axes[1,2].set_ylabel('ELA (m)')
axes[1,2].set_title('Equilibrium Line Altitude')
axes[1,2].grid(True, alpha=0.3)
axes[1,2].legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nFinal Results Comparison:")
print("=" * 50)
print(f"{'Metric':<20} {'T-P Forcing':<15} {'Direct MB':<15}")
print("-" * 50)
print(f"{'Length (km)':<20} {result1.edge[-1]/1000:<15.2f} {result2.edge[-1]/1000:<15.2f}")
print(f"{'Area (km²)':<20} {result1.area[-1]/1e6:<15.2f} {result2.area[-1]/1e6:<15.2f}")
print(f"{'Max thickness (m)':<20} {result1.h[-1,:].max():<15.1f} {result2.h[-1,:].max():<15.1f}")
print(f"{'Final ELA (m)':<20} {result1.ela[-1]:<15.1f} {result2.ela[-1]:<15.1f}")
print(f"{'Mean MB (m/yr)':<20} {(result1.gwb/result1.area)[-100:].mean():<15.3f} {(result2.gwb/result2.area)[-100:].mean():<15.3f}")

## 5. Export Results

Finally, let's show how to export the results to different formats for further analysis.

In [ ]:
# Convert to pandas DataFrame
df1 = result1.to_pandas()
df2 = result2.to_pandas()

print("Pandas DataFrame structure:")
print(df1.head())
print(f"\nDataFrame shape: {df1.shape}")
print(f"Columns: {list(df1.columns)}")

# Convert to xarray Dataset
ds1 = result1.to_xarray()
ds2 = result2.to_xarray()

print("\nXarray Dataset structure:")
print(ds1)

# Save results (uncomment to actually save)
# df1.to_csv('glacier_results_tp_forcing.csv')
# df2.to_csv('glacier_results_mb_forcing.csv')
# ds1.to_netcdf('glacier_results_tp_forcing.nc')
# ds2.to_netcdf('glacier_results_mb_forcing.nc')
# result1.to_pickle('glacier_model_tp_forcing.pkl')
# result2.to_pickle('glacier_model_mb_forcing.pkl')

print("\n✓ Results ready for export!")
print("  - Use .to_pandas() for DataFrame export")
print("  - Use .to_xarray() for NetCDF export")
print("  - Use .to_pickle() for full model state")

## Summary

This notebook demonstrated:

1. **Model Setup**: How to configure the flowline2d model with synthetic geometry
2. **Forcing Methods**: Two different approaches to drive the model:
   - Temperature-precipitation forcing (more physically based)
   - Direct mass balance forcing (more direct control)
3. **Visualization**: Built-in plotting methods and custom comparisons
4. **Data Export**: Multiple formats for further analysis

### Key Features Shown:
- Modular design with separate configuration, geometry, and forcing objects
- Flexible climate forcing options
- Built-in visualization methods
- Multiple export formats (pandas, xarray, pickle)
- Error handling and model validation

### Next Steps:
- Try different parameter values to see their effects
- Experiment with more complex climate scenarios
- Use real glacier geometry data
- Perform sensitivity analysis or calibration studies